In [1]:
%load_ext autoreload
%autoreload all

In [10]:
import sys
from pathlib import Path

# Add the parent directory to sys.path so we can import src
sys.path.insert(0, str(Path().resolve().parent)) 
# Importing necessary libraries and modules
import polars as pl 
from src.pipeline_hero.config import Config, Tokenizer_vocabs
import src.pipeline_hero.tokenizer as tokenizer
from torch.utils.data import Dataset




In [4]:
df_data_by_patient = pl.read_parquet(Config.data_by_patient)

# Sequencer MLM: 1 view, 3 dimensions
- code: [CLS], [gender], [ethnicity], [race], [SEP],[code1] ... [coden]
- day_gap
- segment visit: A,B alternance according to the change


In [6]:
order_seq = ["gender", "ethnicity", "race"]

i = 0

# build code sequence
df = df_data_by_patient.with_columns(
    code_seq = pl.concat_list([
        pl.lit(["[CLS]"]),
        pl.concat_list([pl.col(c) for c in order_seq]),
        pl.lit(["[SEP]"]),
        pl.col("code")
    ])
)

# build segment visit : cls + demo + sep : 3, codes alters 0 and 1
df = df.with_columns(
    pl.col("visit_id")
      .map_elements(build_segment_visit, return_dtype=pl.List(pl.Int8))
      .alias("segment_visit")
)

In [11]:
df_data_by_patient

patient_id,code,day_gap_bd,visit_id,birth_date,race,ethnicity,gender
i64,list[str],list[i64],list[f64],str,str,str,str
115967598,"[""SNOMED/416838001"", ""CPT4/84439"", … ""Domain/OMOP generated""]","[17236, 17236, … 26254]","[null, null, … 2.1664541e8]","""1951-01-20 00:00:00""","""Race/5""","""Ethnicity/Not Hispanic""","""Gender/M"""
115967595,"[""Domain/OMOP generated"", ""Domain/OMOP generated"", … ""RxNorm/314135""]","[12321, 13390, … 18967]","[1.8522471e7, 2.7259934e7, … 2.17352347e8]","""1971-01-29 00:00:00""","""Race/2""","""Ethnicity/Not Hispanic""","""Gender/F"""
115971921,"[""SNOMED/3855007"", ""LOINC/38214-3"", … ""Domain/OMOP generated""]","[26444, 26444, … 28045]","[7.6039659e7, null, … 1.49347368e8]","""1943-03-26 00:00:00""","""Race/3""","""Ethnicity/Not Hispanic""","""Gender/M"""
115969569,"[""SNOMED/417662000"", ""LOINC/72166-2"", … ""CPT4/36415""]","[21128, 21128, … 24596]","[5.6052812e7, 5.6052812e7, … 2.1938286e8]","""1955-09-12 00:00:00""","""Race/5""","""Ethnicity/Not Hispanic""","""Gender/M"""
115968134,"[""LOINC/29463-7"", ""LOINC/8302-2"", … ""Domain/OMOP generated""]","[23340, 23340, … 27193]","[null, null, … 1.79281488e8]","""1946-12-04 00:00:00""","""Race/5""","""Ethnicity/Not Hispanic""","""Gender/F"""
…,…,…,…,…,…,…,…
115967711,"[""SNOMED/427679007"", ""Visit/OP"", … ""SNOMED/713914004""]","[20480, 20794, … 23113]","[null, 1.5305193e8, … 2.17625694e8]","""1959-10-26 00:00:00""","""Race/5""","""Ethnicity/Not Hispanic""","""Gender/M"""
115972299,"[""LOINC/LP173417-9"", ""LOINC/28570-0"", … ""SNOMED/230058003""]","[8748, 8749, … 9702]","[null, 5.2993488e7, … 7.8689376e7]","""1989-03-12 00:00:00""","""Race/5""","""Ethnicity/Not Hispanic""","""Gender/M"""
115970334,"[""LOINC/39156-5"", ""LOINC/29463-7"", … ""SNOMED/228510007""]","[7933, 7933, … 11458]","[1.9053666e7, 1.9053666e7, … 6.5049606e7]","""1983-04-20 00:00:00""","""Race/5""","""Ethnicity/Hispanic""","""Gender/F"""


In [26]:
class PatientMLMDataset(Dataset):
    def __init__(self, data, max_length):
        self.data = data
        # self.tokenizer = tokenizer
        self.max_length = max_length
        self.prefix_fields = ["gender", "ethnicity", "race"]
        self.prefix_len = 1 + len(self.prefix_fields) + 1  # CLS + 3 demo + SEP = 5


    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        code_seq = self.build_code_seq(sample)

        seg_codes = self.build_segment_visit(sample["visit_id"][0])  # length = n_codes
        segment_seq = [0] * self.prefix_len + seg_codes           # length matches code_seq

        day_gap_seq = sample["day_gap_bd"][0]
        day_gap_seq = [0.0] * self.prefix_len + list(day_gap_seq)
        # truncate to max_length (keep prefix + earliest events)
        code_seq = code_seq[: self.max_length]
        segment_seq = segment_seq[: self.max_length]
        day_gap_seq = day_gap_seq[: self.max_length]

        assert len(code_seq) == len(segment_seq) == len(day_gap_seq)

        return {
            "code_seq": code_seq,               # list[str] or list[int]
            "segment_ids": segment_seq,         # list[int]
            "day_gap": day_gap_seq,             # list[float]
        }
    
    def build_code_seq(self, sample):
        return ["[CLS]"] + [sample[c][0] for c in self.prefix_fields] + ["[SEP]"] + list(sample["code"][0])

    def build_segment_visit(self, visit_id_list):
        seg = []
        cur = 0
        prev = None
        first = True
        for v in visit_id_list:
            if not first and v != prev:
                cur = 1 - cur
            seg.append(cur)
            prev = v
            first = False
        return [s + 1 for s in seg]

       


In [27]:
data_loader_mlm = PatientMLMDataset(data = df_data_by_patient , max_length = 10)
data_loader_mlm[0]

{'code_seq': ['[CLS]',
  'Gender/M',
  'Ethnicity/Not Hispanic',
  'Race/5',
  '[SEP]',
  'SNOMED/416838001',
  'CPT4/84439',
  'LOINC/30341-2',
  'LOINC/3024-7',
  'LOINC/42929-0'],
 'segment_ids': [0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
 'day_gap': [0.0, 0.0, 0.0, 0.0, 0.0, 17236, 17236, 17236, 17236, 17236]}